# Phase 2: Dataset Schema & Data Integrity Audit
## AI-Powered Sustainable Transportation Advisor

**Objective:** Inspect the raw Delhi bus Automatic Vehicle Location (AVL) SQLite database schema, evaluate data quality, audit timestamps and time zones, and establish data cleaning and ingestion safeguards.

> **CRITICAL MACHINE LEARNING SAFEGUARD:**  
> Raw AVL records consist of intermittent vehicle pings. **Arbitrary consecutive GPS pings must NOT be treated as ground-truth segment travel times.** True segment travel-time targets must be derived by detecting vehicles crossing defined spatial segment boundaries with verified journey continuity.

### Step 1: Import Standard Libraries & Configure Relative Paths
We use standard Python libraries (`sqlite3`, `pathlib`, `pandas`) configured with paths relative to the project root.

In [1]:
import sqlite3
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Resolve project root relative to the notebooks directory
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

print(f"Project Root  : {PROJECT_ROOT}")
print(f"Raw Data Path : {RAW_DATA_DIR}")

Project Root  : C:\Users\bveda\OneDrive\Desktop\ML-OPS-EL
Raw Data Path : C:\Users\bveda\OneDrive\Desktop\ML-OPS-EL\data\raw


### Step 2: Discover and Connect to a Sample Daily SQLite Database
We look for an extracted daily database (e.g. `2019-07-01.db` or any `*.db` / `*.sqlite` file in `data/raw/`).
We connect in **read-only mode** (`mode=ro`) to prevent accidental modification of raw data.

In [2]:
# Locate candidate SQLite databases
db_files = list(RAW_DATA_DIR.glob("*.db")) + list(RAW_DATA_DIR.glob("*.sqlite"))

if not db_files:
    print("[ℹ️ STATUS] No SQLite database file found in data/raw/ yet.")
    print("Please place a representative daily file (e.g. '2019-07-01.db') into data/raw/.")
    print("See docs/schema_audit_report.md for acquisition instructions.")
    SAMPLE_DB_PATH = None
else:
    SAMPLE_DB_PATH = sorted(db_files)[0]
    print(f"[✅ FOUND] Selected database: {SAMPLE_DB_PATH.name} ({SAMPLE_DB_PATH.stat().st_size / (1024*1024):.2f} MB)")

def get_readonly_connection(db_path):
    """Open SQLite connection in URI read-only mode."""
    if db_path is None or not db_path.exists():
        return None
    uri = f"{db_path.as_uri()}?mode=ro"
    return sqlite3.connect(uri, uri=True)

[✅ FOUND] Selected database: 2019-07-01.db (184.46 MB)


### Step 3: Inspect Tables and Schema (`PRAGMA table_info`)
We inspect table names and column definitions directly from the SQLite catalog without loading entire tables into memory.

In [3]:
if SAMPLE_DB_PATH:
    with get_readonly_connection(SAMPLE_DB_PATH) as conn:
        # List all tables in the SQLite database
        cursor = conn.cursor()
        tables = cursor.execute("SELECT name FROM sqlite_master WHERE type='table';").fetchall()
        print(f"Tables in database: {[t[0] for t in tables]}")
        
        # Detailed schema inspection for 'vehicle_feed'
        schema_info = cursor.execute("PRAGMA table_info(vehicle_feed);").fetchall()
        schema_df = pd.DataFrame(schema_info, columns=["cid", "name", "type", "notnull", "dflt_value", "pk"])
        print("\n--- vehicle_feed Column Schema ---")
        print(schema_df[["cid", "name", "type", "notnull", "pk"]])
else:
    print("[ℹ️ EXPECTED SCHEMA (Verified from Zenodo metadata)]:")
    expected_schema = [
        ("vehicle_id", "TEXT", "Physical bus identifier"),
        ("lat", "TEXT", "Latitude coordinate (stored as text)"),
        ("lng", "TEXT", "Longitude coordinate (stored as text)"),
        ("speed", "REAL", "Instantaneous GPS speed in km/h"),
        ("route_id", "REAL", "Bus route identifier (e.g. 534.0)"),
        ("trip_id", "TEXT", "Active trip identifier"),
        ("timestamp", "TEXT", "Unix epoch timestamp in seconds (UTC)"),
        ("date", "DATE", "Service date in India Standard Time (IST)"),
        ("time", "TEXT", "Service time in India Standard Time (IST, HH:MM:SS)")
    ]
    print(pd.DataFrame(expected_schema, columns=["Column", "Stored SQLite Type", "Description"]))

Tables in database: ['vehicle_feed']

--- vehicle_feed Column Schema ---
   cid        name  type  notnull  pk
0    0  vehicle_id  TEXT        0   0
1    1         lat  TEXT        0   0
2    2         lng  TEXT        0   0
3    3       speed  REAL        0   0
4    4    route_id  REAL        0   0
5    5     trip_id  TEXT        0   0
6    6   timestamp  TEXT        0   0
7    7        date  date        0   0
8    8        time  TEXT        0   0


### Step 4: Record Count & Lightweight Sample Preview
We query the total record count and fetch a small sample (`LIMIT 10`) to verify row contents safely.

In [4]:
if SAMPLE_DB_PATH:
    with get_readonly_connection(SAMPLE_DB_PATH) as conn:
        total_rows = conn.execute("SELECT COUNT(*) FROM vehicle_feed;").fetchone()[0]
        print(f"Total records in {SAMPLE_DB_PATH.name}: {total_rows:,}")
        
        # Preview first 10 rows
        sample_preview = pd.read_sql_query("SELECT * FROM vehicle_feed LIMIT 10;", conn)
        display(sample_preview)
else:
    print("[ℹ️ Awaiting sample database file for row count and preview]")

Total records in 2019-07-01.db: 1,977,922


,vehicle_id,lat,lng,speed,route_id,trip_id,timestamp,date,time
0,DL1PD2284,28.6396675109863,77.2462997436523,1.944444,45.0,1669,1561939202,2019-07-01,05:30:10
1,DL1PC5853,28.7008724212646,77.2187194824219,2.777778,2.0,137,1561939201,2019-07-01,05:30:10
2,DL1PD0891,28.5995502471924,77.2405853271484,0.000000,482.0,14809,1561939200,2019-07-01,05:30:10
3,DL1PD1032,28.6130619049072,76.9962615966797,1.944444,220.0,6044,1561939200,2019-07-01,05:30:10
4,DL1PC6491,28.5722732543945,77.258186340332,1.388889,494.0,15071,1561939200,2019-07-01,05:30:10
5,DL1PD0679,28.6085529327393,76.9570159912109,2.777778,220.0,6046,1561939201,2019-07-01,05:30:10
6,DL1PC6646,28.6478805541992,77.295539855957,1.388889,172.0,4022,1561939204,2019-07-01,05:30:10
7,DL1PC5415,28.5743465423584,77.2289352416992,2.222222,375.0,10392,1561939201,2019-07-01,05:30:10
8,DL1PD0107,28.6291027069092,77.068977355957,2.777778,246.0,6671,1561939200,2019-07-01,05:30:10
9,DL1PD2567,28.6235046386719,77.2012100219727,2.222222,195.0,5131,1561939201,2019-07-01,05:30:10


### Step 5: Missing Value Audit via SQL Aggregation
Rather than loading millions of rows into a Pandas DataFrame, we calculate the count of null/missing entries for every column using in-database SQL aggregation.

In [5]:
if SAMPLE_DB_PATH:
    with get_readonly_connection(SAMPLE_DB_PATH) as conn:
        null_query = """
        SELECT
            COUNT(*) AS total_rows,
            SUM(CASE WHEN vehicle_id IS NULL OR TRIM(vehicle_id) = '' THEN 1 ELSE 0 END) AS missing_vehicle_id,
            SUM(CASE WHEN lat IS NULL OR TRIM(lat) = '' THEN 1 ELSE 0 END) AS missing_lat,
            SUM(CASE WHEN lng IS NULL OR TRIM(lng) = '' THEN 1 ELSE 0 END) AS missing_lng,
            SUM(CASE WHEN speed IS NULL THEN 1 ELSE 0 END) AS missing_speed,
            SUM(CASE WHEN route_id IS NULL THEN 1 ELSE 0 END) AS missing_route_id,
            SUM(CASE WHEN trip_id IS NULL OR TRIM(trip_id) = '' THEN 1 ELSE 0 END) AS missing_trip_id,
            SUM(CASE WHEN timestamp IS NULL OR TRIM(timestamp) = '' THEN 1 ELSE 0 END) AS missing_timestamp,
            SUM(CASE WHEN date IS NULL THEN 1 ELSE 0 END) AS missing_date,
            SUM(CASE WHEN time IS NULL OR TRIM(time) = '' THEN 1 ELSE 0 END) AS missing_time
        FROM vehicle_feed;
        """
        null_stats = pd.read_sql_query(null_query, conn)
        total = null_stats["total_rows"].iloc[0]
        
        missing_summary = []
        for col in null_stats.columns:
            if col != "total_rows":
                cnt = null_stats[col].iloc[0]
                missing_summary.append({
                    "Field": col.replace("missing_", ""),
                    "Missing Count": cnt,
                    "Missing %": (cnt / total) * 100
                })
        missing_df = pd.DataFrame(missing_summary)
        display(missing_df)
else:
    print("[ℹ️ Awaiting sample database file for missingness audit]")

,Field,Missing Count,Missing %
0,vehicle_id,0,0.0
1,lat,0,0.0
2,lng,0,0.0
3,speed,0,0.0
4,route_id,0,0.0
5,trip_id,0,0.0
6,timestamp,0,0.0
7,date,0,0.0
8,time,0,0.0


### Step 6: Coordinate Range & Spatial Bounding Box Audit
Delhi is located approximately between latitude 28.3°N to 28.9°N and longitude 76.8°E to 77.5°E.
Since `lat` and `lng` are stored as `TEXT`, we cast them to `REAL` to audit coordinate boundaries and identify corrupted GPS locations.

In [6]:
if SAMPLE_DB_PATH:
    with get_readonly_connection(SAMPLE_DB_PATH) as conn:
        coord_query = """
        SELECT
            MIN(CAST(lat AS REAL)) AS min_lat,
            MAX(CAST(lat AS REAL)) AS max_lat,
            MIN(CAST(lng AS REAL)) AS min_lng,
            MAX(CAST(lng AS REAL)) AS max_lng,
            SUM(CASE WHEN CAST(lat AS REAL) < 28.3 OR CAST(lat AS REAL) > 28.9 THEN 1 ELSE 0 END) AS out_of_bounds_lat,
            SUM(CASE WHEN CAST(lng AS REAL) < 76.8 OR CAST(lng AS REAL) > 77.5 THEN 1 ELSE 0 END) AS out_of_bounds_lng
        FROM vehicle_feed
        WHERE lat IS NOT NULL AND lng IS NOT NULL;
        """
        coord_stats = pd.read_sql_query(coord_query, conn)
        print("--- Spatial Bounds Check ---")
        display(coord_stats)
else:
    print("[ℹ️ Awaiting sample database file for spatial bounding box check]")

--- Spatial Bounds Check ---


,min_lat,max_lat,min_lng,max_lng,out_of_bounds_lat,out_of_bounds_lng
0,28.490431,28.858192,76.841896,77.341705,0,0


### Step 7: Temporal Alignment & Timezone Audit (UTC Epoch vs. IST)
According to the dataset documentation:
- `timestamp` is the Unix epoch in seconds on a **UTC basis**.
- `date` and `time` are converted to **India Standard Time (IST, UTC+05:30)**.
- The expected difference is 19,800 seconds (5 hours, 30 minutes).

We audit sample records to verify this offset.

In [7]:
if SAMPLE_DB_PATH:
    with get_readonly_connection(SAMPLE_DB_PATH) as conn:
        time_sample = pd.read_sql_query("""
            SELECT vehicle_id, timestamp, date, time
            FROM vehicle_feed
            WHERE timestamp IS NOT NULL AND date IS NOT NULL AND time IS NOT NULL
            LIMIT 5;
        """, conn)
        
        # Convert epoch to UTC and IST datetimes
        time_sample["utc_datetime"] = pd.to_datetime(time_sample["timestamp"].astype(float), unit="s", utc=True)
        time_sample["calculated_ist"] = time_sample["utc_datetime"].dt.tz_convert("Asia/Kolkata")
        time_sample["recorded_ist_str"] = time_sample["date"].astype(str) + " " + time_sample["time"].astype(str)
        
        print("--- Timezone Verification Sample ---")
        display(time_sample[["vehicle_id", "timestamp", "utc_datetime", "calculated_ist", "recorded_ist_str"]])
else:
    print("[ℹ️ Awaiting sample database file for timezone alignment check]")

--- Timezone Verification Sample ---


,vehicle_id,timestamp,utc_datetime,calculated_ist,recorded_ist_str
0,DL1PD2284,1561939202,2019-07-01 00:00:02+00:00,2019-07-01 05:30:02+05:30,2019-07-01 05:30:10
1,DL1PC5853,1561939201,2019-07-01 00:00:01+00:00,2019-07-01 05:30:01+05:30,2019-07-01 05:30:10
2,DL1PD0891,1561939200,2019-07-01 00:00:00+00:00,2019-07-01 05:30:00+05:30,2019-07-01 05:30:10
3,DL1PD1032,1561939200,2019-07-01 00:00:00+00:00,2019-07-01 05:30:00+05:30,2019-07-01 05:30:10
4,DL1PC6491,1561939200,2019-07-01 00:00:00+00:00,2019-07-01 05:30:00+05:30,2019-07-01 05:30:10


### Step 8: Duplicate Observations & Sampling Interval Analysis
We check if duplicate records exist for the same vehicle at the exact same timestamp (`(vehicle_id, timestamp)`), and analyze ping frequencies.

In [8]:
if SAMPLE_DB_PATH:
    with get_readonly_connection(SAMPLE_DB_PATH) as conn:
        dup_query = """
        SELECT COUNT(*) - COUNT(DISTINCT vehicle_id || '_' || timestamp) AS duplicate_pings
        FROM vehicle_feed
        WHERE vehicle_id IS NOT NULL AND timestamp IS NOT NULL;
        """
        dup_result = conn.execute(dup_query).fetchone()[0]
        print(f"Duplicate (vehicle_id, timestamp) pairs: {dup_result:,}")
        
        unique_vehicles = conn.execute("SELECT COUNT(DISTINCT vehicle_id) FROM vehicle_feed;").fetchone()[0]
        unique_routes = conn.execute("SELECT COUNT(DISTINCT route_id) FROM vehicle_feed WHERE route_id IS NOT NULL;").fetchone()[0]
        print(f"Unique Vehicles : {unique_vehicles:,}")
        print(f"Unique Routes   : {unique_routes:,}")
else:
    print("[ℹ️ Awaiting sample database file for duplicate observation check]")

Duplicate (vehicle_id, timestamp) pairs: 73,034


Unique Vehicles : 1,547
Unique Routes   : 476


### Step 9: Audit Summary & Recommendations for Phase 3

#### Summary of Audit Findings:
1. **Schema Standardization:** `lat` and `lng` must be parsed from string to float; `route_id` should be normalized to string integers (`'534'` rather than `'534.0'`).
2. **Filtering Rules Established:**
   - Remove records outside Delhi coordinates ($28.3 \le \text{lat} \le 28.9, 76.8 \le \text{lng} \le 77.5$).
   - Filter out duplicate `(vehicle_id, timestamp)` pairs.
   - Handle trip identifiers when null (some vehicles run without active trip IDs).
3. **Segment Travel Time Derivation (Phase 3 Requirement):**
   - Never calculate segment speed/travel time from arbitrary successive rows.
   - Define spatial entry/exit line boundaries for target route segments.
   - Interpolate exact crossing timestamps for continuous vehicle trajectories.